# SenGuard-MSC · Notebook 02 — Reproduce baselines and benchmark on development data

G2 on GSE284666 only. **No MSC cohort is read in this notebook except the BMMSC cell type, which is part of SenCat itself.** It ends by freezing weights and a model choice under the rule in `ANALYSIS_PROTOCOL.md`.

Sections: confounding structure, published-score reproduction, feature-selection leakage, hUSI and pathway scores, nested leave-one-cell-type-out benchmark, model freeze.

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
ROOT_GUESS = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, CKPT, DEV
import sklearn, xgboost
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)
CKPT.mkdir(exist_ok=True)
sg.require(PROC / f'{DEV}_labels.csv', PROC / f'{DEV}_norm_counts.tsv.gz', REPORTS / 'ANALYSIS_PROTOCOL.md')
print('sklearn', sklearn.__version__, '| xgboost', xgboost.__version__, '| seed', sg.SEED, '| protocol', sg.sha256(REPORTS / 'ANALYSIS_PROTOCOL.md')[:16])

labels = pd.read_csv(PROC / f'{DEV}_labels.csv').set_index('sample_id')
counts_all, ann = sg.load_as_deposited(DEV)
counts_all = counts_all.loc[labels.index]
norm = pd.read_csv(PROC / f'{DEV}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[labels.index]
y, groups = labels.y.to_numpy(), labels.cell_line.to_numpy()
print('counts', counts_all.shape, '| norm', norm.shape, '| positives', int(y.sum()), '| cell types', len(np.unique(groups)))

## Confounding structure [measured]
How much expression variance follows cell type versus senescence label, from a PCA on log-normalized genes. PCA here is descriptive and fits nothing used later.

In [ ]:
from sklearn.decomposition import PCA
xlog = np.log1p(norm)
top = xlog.var().nlargest(5000).index
Z = xlog[top] - xlog[top].mean()
pca = PCA(n_components=10, random_state=sg.SEED).fit(Z)
pcs = pca.transform(Z)
def r2(pc, factor):
    d = pd.DataFrame({'pc': pc, 'f': factor})
    return 1 - d.groupby('f').pc.transform(lambda v: v - v.mean()).pow(2).sum() / (d.pc - d.pc.mean()).pow(2).sum()
paradigm = labels.treatment.str.replace(r'\s*\(.*', '', regex=True).to_numpy()
conf = pd.DataFrame({'PC': np.arange(1, 11), 'var_explained': pca.explained_variance_ratio_, 'r2_cell_type': [r2(pcs[:, i], groups) for i in range(10)],
                     'r2_senescence_label': [r2(pcs[:, i], y) for i in range(10)], 'r2_treatment': [r2(pcs[:, i], paradigm) for i in range(10)]}).round(3)
conf.to_csv(RESULTS / 'dev_confounding_pca.csv', index=False)
print(conf.to_string(index=False))
print('\nLabel is balanced within cell types by design, but treatment dose and type differ by cell type (see manifest), and no donor information exists.')

## Reproducing the published SenCat score
1. **Coefficient reproduction:** refit `ml_classifier.py` at config `loose5000f` on GEO counts and compare with the repo's coefficients. The authors' input was a supplementary table not in GEO, so an exact match isn't guaranteed.
2. **Score:** published weights on median-of-ratios counts, log1p applied once. These weights were fit on these same samples, so this measures self-consistency and is **not** validation.

In [ ]:
pub = sg.load_sencat_markers()
pub_coef = pub.coef
t0 = time.time()
re_coef, re_folds = sg.author_pipeline(norm, y, groups)
print(f'Refit in {time.time() - t0:.1f}s: {len(re_coef)} features')
shared = re_coef.index.intersection(pub_coef.index)
from scipy import stats
repro = {'published_features': len(pub_coef), 'refit_features': len(re_coef), 'overlap': len(shared), 'overlap_share_of_published': len(shared) / len(pub_coef),
         'spearman_coef': float(stats.spearmanr(pub_coef.loc[shared], re_coef.loc[shared]).statistic), 'pearson_coef': float(np.corrcoef(pub_coef.loc[shared], re_coef.loc[shared])[0, 1]),
         'sign_agreement': float((np.sign(pub_coef.loc[shared]) == np.sign(re_coef.loc[shared])).mean())}
print(json.dumps(repro, indent=1))

pub_score, n_used = sg.weighted_log1p_score(norm, pub_coef)
refit_score, _ = sg.weighted_log1p_score(norm, re_coef)
print(f'Published weights: {n_used}/{len(pub_coef)} features in the filtered matrix. Correlation published vs refit score: {np.corrcoef(pub_score, refit_score)[0, 1]:.4f}')
pub_auc = sg.score_by_group_auc(pub_score.to_numpy(), y, groups)
print(pub_auc.round(3).to_string(index=False))
print('Published score, mean within-cell-type AUROC (self-consistency, not validation): %.3f' % pub_auc.auroc.mean())

## Measuring feature-selection leakage in the authors' design
`ml_classifier.py` runs `SelectKBest` on all samples before leave-one-cell-type-out. The comparison below uses the same logistic model and the same `k`:
- **as published:** selection sees the held-out cell type,
- **clean:** selection, gene filters and the normalization reference use training cell types only.

The score in each fold comes from that fold's model, so it is out-of-sample for the fit in both arms.

In [ ]:
def author_loco_scores(counts, y, groups, clean, k=5000):
    rows = []
    if not clean:
        n_all = sg.mor_normalize(sg.author_gene_filter(counts))[0]
        pre = np.log1p(sg.author_feature_prefilter(n_all))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            cols = pre.columns[sg.SelectKBest(sg.f_classif, k=min(k, pre.shape[1])).fit(pre.to_numpy(), y).get_support(indices=True)]
    for g in np.unique(groups):
        tr = groups != g
        if clean:
            genes, ref = sg.prefilter_train(counts.loc[tr])
            Xtr, Xte = sg.fold_transform(counts.loc[tr], genes, ref), sg.fold_transform(counts.loc[~tr], genes, ref)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                use = Xtr.columns[sg.SelectKBest(sg.f_classif, k=min(k, Xtr.shape[1])).fit(Xtr.to_numpy(), y[tr]).get_support(indices=True)]
            Xtr, Xte = Xtr[use], Xte[use]
        else:
            Xtr, Xte = pre.loc[tr, cols], pre.loc[~tr, cols]
        m = sg.LogisticRegression(solver='liblinear').fit(Xtr.to_numpy(), y[tr])
        s = m.decision_function(Xte.to_numpy())
        rows.append({'group': g, 'auroc': sg.roc_auc_score(y[~tr], s), 'auprc': sg.average_precision_score(y[~tr], s), 'n': int((~tr).sum())})
    return pd.DataFrame(rows)

leak = author_loco_scores(counts_all, y, groups, clean=False).merge(author_loco_scores(counts_all, y, groups, clean=True), on=['group', 'n'], suffixes=('_as_published', '_clean'))
leak['difference'] = leak.auroc_as_published - leak.auroc_clean
leak.to_csv(RESULTS / 'dev_feature_selection_leakage.csv', index=False)
print(leak.round(3).to_string(index=False))
print('Mean AUROC as published %.4f | clean %.4f | mean difference %.4f' % (leak.auroc_as_published.mean(), leak.auroc_clean.mean(), leak.difference.mean()))

## hUSI and simple pathway scores
- **hUSI:** per-sample Spearman with published weights. Its AUROC is unaffected by the authors' min-max step within a cell type only when computed on raw Spearman, so raw values are used. These weights were trained on other public data, so SenCat is an out-of-training test for this score, assuming no sample overlap, which `trainSet_meta843.csv` is checked for below.
- **Pathway scores:** rank-based and fitted to nothing: SenMayo, Hallmark E2F and G2M (expected to fall), canonical five-marker transcripts from the authors' config.

In [ ]:
h, extra = sg.load_gene_sets()
sym_ann = ann.reindex(norm.columns)
ens2sym = pd.read_csv(PROC / 'GSE309666_matrix_as_deposited.tsv.gz', sep='\t', usecols=['ensembl', 'symbol'], dtype=str).dropna().drop_duplicates('ensembl').set_index('ensembl').symbol
ann_sym = pd.DataFrame({'symbol': ens2sym.reindex(norm.columns)}, index=norm.columns)
print('Ensembl-to-symbol map is annotation only (GSE309666 file columns gene_id and gene_name); genes mapped:', int(ann_sym.symbol.notna().sum()), 'of', norm.shape[1])
norm_sym = sg.to_symbols(norm, ann_sym)
husi_w = sg.load_husi_weights()
husi, n_h = sg.husi_raw(norm_sym, husi_w)
print(f'hUSI features used: {n_h}/{len(husi_w)}')
meta = pd.read_csv(sg.EXT / 'HUSI/Data/trainSet_meta843.csv', low_memory=False, encoding='latin-1')  # file is not UTF-8
txt = meta.astype(str).agg(' '.join, axis=1)
print('hUSI training metadata rows', len(meta), '| columns', list(meta.columns)[:8])
print('  Mentions of the three cohort accessions:', {a: int(txt.str.contains(a).sum()) for a in ['GSE284666', 'GSE309666', 'GSE178514']})
print('  Mentions of MSC or mesenchymal:', int(txt.str.contains(r'mesenchym|\bMSC', case=False, regex=True).sum()))

scores = {'SenCat published (self-consistency)': pub_score, 'hUSI raw': husi}
n_sets = {}
for name, genes, sign in [('SenMayo', extra['SenMayo'], 1), ('Hallmark E2F (inverted)', h['HALLMARK_E2F_TARGETS'], -1), ('Hallmark G2M (inverted)', h['HALLMARK_G2M_CHECKPOINT'], -1)]:
    s, n = sg.rank_set_score(norm_sym, genes)
    scores[name] = sign * s
    n_sets[name] = n
canon = pd.read_csv(sg.EXT / 'wf-ml-markers-senescence/config/Canonical_gene_markers.csv', index_col=0).gene
print('Gene-set coverage', n_sets)
can_scores = {}
for ens, g in canon.items():
    if ens in norm.columns:
        s = np.log1p(norm[ens])
        can_scores[f'{g}{" (inverted)" if g == "LMNB1" else ""}'] = -s if g == 'LMNB1' else s
scores.update(can_scores)
simple_rows, per_type = [], {}
for name, s in scores.items():
    a = sg.score_by_group_auc(s.to_numpy(), y, groups)
    per_type[name] = a.set_index('group').auroc
    m, lo, hi = sg.bootstrap_mean_ci(a.auroc)
    simple_rows.append({'score': name, 'mean_within_type_auroc': m, 'ci_low': lo, 'ci_high': hi, 'min_type_auroc': a.auroc.min(), 'pooled_auroc': sg.roc_auc_score(y, s), 'BMMSC': per_type[name]['BMMSC'], 'prAD': per_type[name]['prAD']})
simple = pd.DataFrame(simple_rows).sort_values('mean_within_type_auroc', ascending=False)
simple.to_csv(RESULTS / 'dev_simple_scores.csv', index=False)
print(simple.round(3).to_string(index=False))
pd.DataFrame(scores).rename_axis('sample_id').to_csv(RESULTS / 'dev_simple_scores_per_sample.csv')

## Nested leave-one-cell-type-out benchmark [measured]
All 14 cell types take turns as the test set. In each outer fold the reference, gene filters, selection, hyperparameters and threshold come from training cell types, with inner leave-one-cell-type-out tuning.

XGBoost uses the GPU when one works, at the project lead's request; its speed and CPU/GPU agreement on a real fold are printed so the choice stays auditable. scikit-learn models have no GPU path and run in parallel on CPU; XGBoost folds run one at a time to stay within VRAM. Results are checkpointed, so a rerun loads them instead of refitting.

In [ ]:
ckpt = CKPT / 'nested_loco_results.json'
held = 'BMMSC'
tr = groups != held
g_genes, g_ref = sg.prefilter_train(counts_all.loc[tr])
Xs = sg.fold_transform(counts_all.loc[tr], g_genes, g_ref)
Xs = Xs[Xs.var().nlargest(2000).index].to_numpy()
Xs_te = sg.fold_transform(counts_all.loc[~tr], g_genes, g_ref)[sg.fold_transform(counts_all.loc[tr], g_genes, g_ref).var().nlargest(2000).index].to_numpy()
speed, xgb_preds = {}, {}
for dev in ['cpu', 'cuda']:
    try:
        t0 = time.time()
        m = xgboost.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, tree_method='hist', device=dev, n_jobs=4, random_state=sg.SEED, eval_metric='logloss').fit(Xs, y[tr])
        speed[dev] = round(time.time() - t0, 2)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            xgb_preds[dev] = m.predict_proba(Xs_te)[:, 1]
    except Exception as e:
        speed[dev] = f'failed: {type(e).__name__}: {e}'
print('XGBoost fit seconds on one real fold:', speed)
xgb_device = 'cuda' if isinstance(speed.get('cuda'), float) else 'cpu'
if len(xgb_preds) == 2:
    print('CPU vs GPU on the held-out fold: max abs probability difference %.5f; AUROC %.4f vs %.4f' % (np.abs(xgb_preds['cpu'] - xgb_preds['cuda']).max(), sg.roc_auc_score(y[~tr], xgb_preds['cpu']), sg.roc_auc_score(y[~tr], xgb_preds['cuda'])))
print('Device used for XGBoost:', xgb_device)

if ckpt.exists():
    results = json.loads(ckpt.read_text())
    print('Loaded checkpoint', ckpt.name, len(results), 'fold results')
else:
    t0 = time.time()
    grid = sg.model_grid(xgb_device)
    cpu_ckpt = CKPT / 'nested_loco_cpu_models.json'
    if cpu_ckpt.exists():
        cpu_results = json.loads(cpu_ckpt.read_text())
    else:
        cpu_results = sg.run_nested_loco(counts_all, y, groups, {k: v for k, v in grid.items() if k != 'xgboost'}, n_jobs=max(1, (os.cpu_count() or 2) - 2))
        cpu_ckpt.write_text(json.dumps(cpu_results))
    print(f'scikit-learn models on CPU: {(time.time() - t0) / 60:.1f} min')
    t1 = time.time()
    xgb_results = sg.run_nested_loco(counts_all, y, groups, {'xgboost': grid['xgboost']}, n_jobs=1 if xgb_device == 'cuda' else max(1, (os.cpu_count() or 2) - 2))
    print(f'XGBoost on {xgb_device}: {(time.time() - t1) / 60:.1f} min')
    results = cpu_results + xgb_results
    ckpt.write_text(json.dumps(results))
    print(f'Fit {len(results)} outer folds in {(time.time() - t0) / 60:.1f} min')

In [ ]:
rows, preds = [], []
for r in results:
    m = sg.fold_metrics(r['y'], r['score'], r['threshold'])
    rows.append({'model': r['model'], 'held_out': r['held_out'], **m, 'threshold': r['threshold'], 'inner_auc': r['inner_auc'], 'best_params': json.dumps(r['best_params'])})
    preds += [{'model': r['model'], 'held_out': r['held_out'], 'sample_id': s, 'y': a, 'score': b, 'threshold_from_training': r['threshold'], 'call': int(b >= r['threshold'])} for s, a, b in zip(r['sample_id'], r['y'], r['score'])]
fold_df = pd.DataFrame(rows)
pred_df = pd.DataFrame(preds)
fold_df.to_csv(RESULTS / 'dev_nested_loco_folds.csv', index=False)
pred_df.to_csv(RESULTS / 'dev_nested_loco_predictions.csv', index=False)
pd.DataFrame({'sample_id': labels.index, 'fold_held_out_cell_type': groups, 'y': y}).to_csv(RESULTS / 'dev_fold_assignments.csv', index=False)

ORDER = ['logreg_l1', 'elastic_net', 'linear_svm', 'rbf_svm', 'random_forest', 'xgboost']
summ = []
for name in ORDER:
    f = fold_df[fold_df.model == name]
    m, lo, hi = sg.bootstrap_mean_ci(f.auroc)
    bm, bl, bh = sg.bootstrap_mean_ci(f.balanced_acc)
    summ.append({'model': name, 'mean_auroc': m, 'ci_low': lo, 'ci_high': hi, 'min_auroc': f.auroc.min(), 'worst_type': f.loc[f.auroc.idxmin(), 'held_out'], 'mean_auprc': f.auprc.mean(), 'balanced_acc': bm, 'ba_ci_low': bl, 'ba_ci_high': bh,
                 'macro_f1': f.macro_f1.mean(), 'BMMSC_auroc': f[f.held_out == 'BMMSC'].auroc.iloc[0], 'prAD_auroc': f[f.held_out == 'prAD'].auroc.iloc[0]})
summary = pd.DataFrame(summ)
summary.to_csv(RESULTS / 'dev_model_summary.csv', index=False)
print(summary.round(3).to_string(index=False))
print('\nAUROC by held-out cell type:')
wide = fold_df.pivot(index='held_out', columns='model', values='auroc')[ORDER]
wide.to_csv(RESULTS / 'dev_auroc_by_cell_type.csv')
print(wide.round(3).to_string())
print('\nChosen hyperparameters per model (count over 14 folds):')
for name in ORDER:
    print(' ', name, fold_df[fold_df.model == name].best_params.value_counts().to_dict())

## Which conditions are hard? [measured]
Near-perfect AUROC within most cell types raises the question of what is actually being separated. This breaks held-out errors down by condition, using the chosen threshold from training folds, and compares each condition with its own cell type's controls. A control-versus-control row checks that vector control and proliferating control aren't themselves separable.

In [ ]:
meta_p = labels.assign(condition=labels.treatment.str.replace(r'\s*\(.*', '', regex=True))
rows = []
for name in ORDER:
    f = pred_df[pred_df.model == name].merge(meta_p[['condition', 'cell_line']], left_on='sample_id', right_index=True)
    for (ct, cond), g in f[f.y == 1].groupby(['cell_line', 'condition']):
        ctrl = f[(f.cell_line == ct) & (f.y == 0)]
        rows.append({'model': name, 'cell_line': ct, 'condition': cond, 'n': len(g), 'auroc_vs_own_controls': sg.roc_auc_score([1] * len(g) + [0] * len(ctrl), list(g.score) + list(ctrl.score))})
by_cond = pd.DataFrame(rows)
by_cond.to_csv(RESULTS / 'dev_auroc_by_condition.csv', index=False)
print(by_cond[by_cond.auroc_vs_own_controls < 1].round(3).to_string(index=False))
print('\nMean AUROC by condition across models and cell types:')
print(by_cond.groupby('condition').auroc_vs_own_controls.agg(['mean', 'min', 'count']).round(3).to_string())
pub_cond = []
for (ct, cond), g in meta_p[meta_p.y == 1].groupby(['cell_line', 'condition']):
    c = meta_p[(meta_p.cell_line == ct) & (meta_p.y == 0)]
    for nm, sc in [('published SenCat', pub_score), ('hUSI raw', husi), ('Hallmark E2F (inverted)', scores['Hallmark E2F (inverted)'])]:
        pub_cond.append({'score': nm, 'cell_line': ct, 'condition': cond, 'auroc': sg.roc_auc_score([1] * len(g) + [0] * len(c), list(sc[g.index]) + list(sc[c.index]))})
pub_cond = pd.DataFrame(pub_cond)
print('\nFixed scores by condition:')
print(pub_cond.pivot_table(index='condition', columns='score', values='auroc').round(3).to_string())
for ct in ['BJ', 'WI-38']:
    i = meta_p.cell_line == ct
    a = meta_p[i & meta_p.condition.isin(['Empty vector'])].index
    b = meta_p[i & meta_p.condition.str.startswith('Control')].index
    print(f'{ct}: published score, proliferating control mean {pub_score[b].mean():.1f} vs empty vector {pub_score[a].mean():.1f}; RasV12 mean {pub_score[meta_p[i & (meta_p.condition == "RasV12")].index].mean():.1f}')

## Calibration, within the development set only
Pooled out-of-fold probabilities for the two tree models and logistic models. They come from different fold models, and calibration on cell types is the question that matters, so these numbers describe held-out cell types. They say nothing about MSC passage samples.

In [ ]:
from sklearn.calibration import calibration_curve
cal_rows = []
for name in ['logreg_l1', 'elastic_net', 'random_forest', 'xgboost']:
    f = pred_df[pred_df.model == name]
    p = 1 / (1 + np.exp(-f.score)) if name in ('logreg_l1', 'elastic_net') else f.score
    frac, mean_pred = calibration_curve(f.y, p, n_bins=5, strategy='quantile')
    cal_rows.append({'model': name, 'brier': sg.brier_score_loss(f.y, p), 'base_rate_brier': f.y.mean() * (1 - f.y.mean()), 'max_bin_gap': float(np.abs(frac - mean_pred).max())})
cal = pd.DataFrame(cal_rows)
cal.to_csv(RESULTS / 'dev_calibration.csv', index=False)
print(cal.round(3).to_string(index=False))

## Freeze: model choice and weights, before notebook 03 reads any MSC score
Applies the rule in the protocol. Three weight sets are fit from SenCat using the authors' pipeline shape, plus the chosen model on all cell types:
- `all14`: all cell types,
- `no_BMMSC`: for cell-type-independent BM-MSC transport,
- `no_BMMSC_prAD`: also drops the adipose-lineage preadipocytes for AD-MSC.

Published weights remain the primary score.

In [ ]:
best_auc = summary.mean_auroc.max()
tied = summary[summary.mean_auroc >= best_auc - 0.01]
chosen = next(m for m in ORDER if m in set(tied.model))
print('Best mean AUROC %.4f; tied within 0.01: %s; chosen by simplicity order: %s' % (best_auc, tied.model.tolist(), chosen))
params = json.loads(fold_df[fold_df.model == chosen].best_params.value_counts().index[0])
print('Most frequent hyperparameters across folds:', params)

def fit_refit_weights(mask, tag):
    c = counts_all.loc[mask]
    filt = sg.author_gene_filter(c)
    n, _, ref = sg.mor_normalize(filt)
    coef, _ = sg.author_pipeline(n, y[mask], groups[mask])
    path = RESULTS / f'frozen_weights_refit_{tag}.csv'
    coef.rename_axis('gene_id').reset_index().to_csv(path, index=False)
    return path, ref

frozen = {}
pub_path = sg.EXT / 'wf-ml-markers-senescence/analysis/transcriptomics_loose5000f_tuned_common_features.csv'
frozen['published_sencat'] = {'path': str(pub_path), 'sha256': sg.sha256(pub_path)}
masks = {'all14': np.ones(len(y), bool), 'no_BMMSC': groups != 'BMMSC', 'no_BMMSC_prAD': ~np.isin(groups, ['BMMSC', 'prAD'])}
for tag, mask in masks.items():
    path, ref = fit_refit_weights(mask, tag)
    frozen[f'refit_{tag}'] = {'path': str(path), 'sha256': sg.sha256(path), 'n_train': int(mask.sum()), 'cell_types': sorted(set(groups[mask]))}
    print(tag, 'trained on', int(mask.sum()), 'samples, weights', pd.read_csv(path).shape[0])

import joblib
pipe, _ = sg.model_grid(xgb_device)[chosen]
model_rows = {}
for tag, mask in masks.items():
    genes, ref = sg.prefilter_train(counts_all.loc[mask])
    Xall = sg.fold_transform(counts_all.loc[mask], genes, ref)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fitted = sg.clone(pipe).set_params(**params).fit(Xall.to_numpy(), y[mask])
    path = CKPT / f'frozen_{chosen}_{tag}.joblib'
    joblib.dump({'model': fitted, 'genes': genes, 'ref': ref, 'name': chosen, 'params': params, 'tag': tag}, path)
    frozen[f'{chosen}_{tag}'] = {'path': str(path), 'sha256': sg.sha256(path), 'n_train': int(mask.sum())}

freeze = {'frozen_utc': pd.Timestamp.now('UTC').isoformat(), 'chosen_model': chosen, 'params': params, 'selection': {'best_mean_auroc': float(best_auc), 'tied': tied.model.tolist()},
          'protocol_sha256': sg.sha256(REPORTS / 'ANALYSIS_PROTOCOL.md'), 'artifacts': frozen, 'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__, 'seed': sg.SEED}
(RESULTS / 'frozen_model.json').write_text(json.dumps(freeze, indent=1))
print('Frozen. sha256 of frozen_model.json:', sg.sha256(RESULTS / 'frozen_model.json'))

## Figure 1: held-out performance by cell type
Dots are per-cell-type AUROC, with a bootstrap CI over cell types on the mean. MSC-relevant types are marked. Values and their table are in `results/dev_auroc_by_cell_type.csv`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={'width_ratios': [1.15, 1]})
ax = axes[0]
disp = {'logreg_l1': 'L1 logistic', 'elastic_net': 'Elastic net', 'linear_svm': 'Linear SVM', 'rbf_svm': 'RBF SVM', 'random_forest': 'Random forest', 'xgboost': 'XGBoost'}
ypos = np.arange(len(ORDER))[::-1]
for yy, name in zip(ypos, ORDER):
    f = fold_df[fold_df.model == name]
    rng = np.random.default_rng(1)
    other = f[~f.held_out.isin(['BMMSC', 'prAD'])]
    ax.scatter(other.auroc, yy + rng.uniform(-0.18, 0.18, len(other)), s=26, color=sg.PAL['muted'], alpha=0.7, linewidths=0, label='Other cell types' if yy == ypos[0] else None, zorder=2)
    for t, c, lab in [('BMMSC', sg.PAL['blue'], 'BMMSC held out'), ('prAD', sg.PAL['orange'], 'Preadipocytes held out')]:
        ax.scatter(f[f.held_out == t].auroc, [yy], s=60, color=c, edgecolors=sg.PAL['surface'], linewidths=1.5, label=lab if yy == ypos[0] else None, zorder=4)
    s = summary[summary.model == name].iloc[0]
    ax.plot([s.ci_low, s.ci_high], [yy - 0.34, yy - 0.34], color=sg.PAL['ink'], lw=2, solid_capstyle='round')
    ax.scatter([s.mean_auroc], [yy - 0.34], s=34, color=sg.PAL['ink'], zorder=5, label='Mean, 95% CI over types' if yy == ypos[0] else None)
ax.set_yticks(ypos, [disp[m] for m in ORDER])
ax.set_xlabel('AUROC on the held-out cell type')
ax.set_title('Nested leave-one-cell-type-out')
ax.set_ylim(-0.75, len(ORDER) - 0.4)
ax.grid(axis='y', visible=False)
ax.legend(loc='lower left', fontsize=8)

ax = axes[1]
show = simple[~simple.score.str.contains('self-consistency')].sort_values('mean_within_type_auroc')
yy = np.arange(len(show))
ax.hlines(yy, show.ci_low, show.ci_high, color=sg.PAL['blue'], lw=2)
ax.scatter(show.mean_within_type_auroc, yy, s=40, color=sg.PAL['blue'], zorder=3, edgecolors=sg.PAL['surface'], linewidths=1.5)
ax.axvline(0.5, color=sg.PAL['muted'], lw=1, ls=(0, (3, 3)))
ax.set_yticks(yy, show.score, fontsize=8.5)
ax.set_xlabel('Mean within-cell-type AUROC, 95% CI')
ax.set_title('Scores that fit nothing on SenCat')
ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(sg.FIGS / 'fig1_development_performance.png')
plt.show()

## G2 report

In [ ]:
best = summary[summary.model == chosen].iloc[0]
report = f"""# G2: baselines on development data

Status: see the summary below. Data: GSE284666, {len(y)} samples, {int(y.sum())} senescence-induced, {len(np.unique(groups))} cell types. Validation unit: cell type. No donor information exists, so nothing here is donor-independent.

## Published score reproduction [measured]
- Coefficient refit vs published: {repro['overlap']}/{repro['published_features']} shared features, Spearman {repro['spearman_coef']:.3f}, sign agreement {repro['sign_agreement']:.3f}.
- The authors' exact input table is not in GEO, so an exact match was not expected.
- Published score mean within-cell-type AUROC {pub_auc.auroc.mean():.3f}: self-consistency only, because those weights were fit on these samples.

## Feature-selection leakage [measured]
As published {leak.auroc_as_published.mean():.4f} vs clean {leak.auroc_clean.mean():.4f}; mean difference {leak.difference.mean():.4f}.

## Benchmark, nested leave-one-cell-type-out [measured]
{sg.pd.DataFrame(summary).round(3).to_string(index=False)}

Chosen under the protocol rule: {chosen}, params {params}.

## Scores that fit nothing on SenCat [measured]
{simple.round(3).to_string(index=False)}

## Confounding
Variance by PC is in results/dev_confounding_pca.csv. Dose and duration differ by cell type, and paradigms are not crossed with cell type. Replicates sit within a cell line, so per-sample counts overstate independent evidence.
"""
(REPORTS / 'G2_BASELINES.md').write_text(report, encoding='utf-8')
print(report)
outs = [RESULTS / f for f in ['dev_confounding_pca.csv', 'dev_feature_selection_leakage.csv', 'dev_simple_scores.csv', 'dev_simple_scores_per_sample.csv', 'dev_nested_loco_folds.csv', 'dev_nested_loco_predictions.csv',
                              'dev_fold_assignments.csv', 'dev_model_summary.csv', 'dev_auroc_by_cell_type.csv', 'dev_calibration.csv', 'frozen_model.json']] + [sg.FIGS / 'fig1_development_performance.png', REPORTS / 'G2_BASELINES.md']
print(sg.output_manifest(outs).assign(path=lambda d: d.path.str.replace(str(ROOT), '', regex=False))[['path', 'bytes']].to_string())